In [2]:
!pip install seaborn


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
    confusion_matrix
)

In [ ]:
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

PROJECT_ROOT = Path("../")

TEST_RGB_DIR = (
    PROJECT_ROOT /
    "dataset/processed/STRUM/test/images"
)

TEST_MASK_DIR = (
    PROJECT_ROOT /
    "dataset/processed/STRUM/test/masks"
)

TEST_SAM_DIR = (
    PROJECT_ROOT /
    "dataset/sam_embeddings/test"
)

BASELINE_WEIGHT = (
    PROJECT_ROOT /
    "checkpoints/best_unet_weighted_augmented.pth"
)

HYBRID_WEIGHT = (
    PROJECT_ROOT /
    "checkpoints/best_hybrid_sam_unet.pth"
)

print("Device:", DEVICE)

Device: cuda


In [17]:
from torch.utils.data import Dataset

In [18]:
class EvaluationDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        sam_dir
    ):

        self.images = sorted(
            Path(image_dir).glob("*.npy")
        )

        self.masks = sorted(
            Path(mask_dir).glob("*.npy")
        )

        self.sam_features = sorted(
            Path(sam_dir).glob("*.npy")
        )

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        image = np.load(
            self.images[idx]
        )

        mask = np.load(
            self.masks[idx]
        )

        sam_feature = np.load(
            self.sam_features[idx]
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        )

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        ).unsqueeze(0)

        sam_feature = torch.tensor(
            sam_feature,
            dtype=torch.float32
        )

        return (
            image,
            mask,
            sam_feature
        )

In [19]:
from torch.utils.data import DataLoader

In [21]:
test_dataset = EvaluationDataset(
    TEST_RGB_DIR,
    TEST_MASK_DIR,
    TEST_SAM_DIR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print(
    "Test Samples:",
    len(test_dataset)
)

Test Samples: 268


In [28]:
class DoubleConvUNet(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):

        return self.conv(x)

In [29]:
class UNet(nn.Module):

    def __init__(
        self,
        in_channels=3,
        out_channels=1
    ):

        super().__init__()

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Encoder

        self.down1 = DoubleConvUNet(
            in_channels,
            64
        )

        self.down2 = DoubleConvUNet(
            64,
            128
        )

        self.down3 = DoubleConvUNet(
            128,
            256
        )

        self.down4 = DoubleConvUNet(
            256,
            512
        )

        # Bottleneck

        self.bottleneck = DoubleConvUNet(
            512,
            1024
        )

        # Decoder

        self.up4 = nn.ConvTranspose2d(
            1024,
            512,
            kernel_size=2,
            stride=2
        )

        self.dec4 = DoubleConvUNet(
            1024,
            512
        )

        self.up3 = nn.ConvTranspose2d(
            512,
            256,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConvUNet(
            512,
            256
        )

        self.up2 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConvUNet(
            256,
            128
        )

        self.up1 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConvUNet(
            128,
            64
        )

        self.final = nn.Conv2d(
            64,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):

        d1 = self.down1(x)

        d2 = self.down2(
            self.pool(d1)
        )

        d3 = self.down3(
            self.pool(d2)
        )

        d4 = self.down4(
            self.pool(d3)
        )

        bottleneck = self.bottleneck(
            self.pool(d4)
        )

        u4 = self.up4(
            bottleneck
        )

        u4 = torch.cat(
            [u4, d4],
            dim=1
        )

        u4 = self.dec4(u4)

        u3 = self.up3(u4)

        u3 = torch.cat(
            [u3, d3],
            dim=1
        )

        u3 = self.dec3(u3)

        u2 = self.up2(u3)

        u2 = torch.cat(
            [u2, d2],
            dim=1
        )

        u2 = self.dec2(u2)

        u1 = self.up1(u2)

        u1 = torch.cat(
            [u1, d1],
            dim=1
        )

        u1 = self.dec1(u1)

        return self.final(u1)

In [30]:
class DoubleConvHybrid(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):

        return self.block(x)


class HybridSAMUNet(nn.Module):
    def __init__(self, in_channels: int = 3, sam_channels: int = 256):
        super().__init__()
        self.e1 = DoubleConvHybrid(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.e2 = DoubleConvHybrid(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConvHybrid(128 + sam_channels, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d1 = DoubleConvHybrid(128 + 128, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d2 = DoubleConvHybrid(64 + 64, 64)
        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x, sam_feature, return_shapes: bool = False):
        e1 = self.e1(x)
        p1 = self.pool1(e1)
        e2 = self.e2(p1)
        p2 = self.pool2(e2)

        raw_sam_shape = tuple(sam_feature.shape)
        sam_resized = F.interpolate(
            sam_feature,
            size=p2.shape[2:],
            mode="bilinear",
            align_corners=False,
        )

        fusion = torch.cat([p2, sam_resized], dim=1)
        b = self.bottleneck(fusion)

        u1 = self.up1(b)
        u1 = torch.cat([u1, e2], dim=1)
        d1 = self.d1(u1)

        u2 = self.up2(d1)
        u2 = torch.cat([u2, e1], dim=1)
        d2 = self.d2(u2)
        logits = self.out(d2)

        if return_shapes:
            return logits, {
                "encoder_p2_shape": tuple(p2.shape),
                "sam_feature_shape": raw_sam_shape,
                "resized_sam_feature_shape": tuple(sam_resized.shape),
                "fusion_tensor_shape": tuple(fusion.shape),
                "output_shape": tuple(logits.shape),
                "resize_strategy": "torch.nn.functional.interpolate(mode='bilinear', align_corners=False)",
            }
        return logits

model = HybridSAMUNet(in_channels=3, sam_channels=256).to(DEVICE)
print(model.__class__.__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))

HybridSAMUNet
Parameters: 2453953


In [31]:
baseline_model = UNet().to(DEVICE)

baseline_model.load_state_dict(
    torch.load(
        BASELINE_WEIGHT,
        map_location=DEVICE
    )
)

baseline_model.eval()

print("Baseline Loaded")

Baseline Loaded


In [32]:
hybrid_checkpoint = torch.load(
    HYBRID_WEIGHT,
    map_location=DEVICE
)

In [33]:
hybrid_model = HybridSAMUNet(
    in_channels=3,
    sam_channels=256
).to(DEVICE)

hybrid_model.load_state_dict(
    hybrid_checkpoint[
        "model_state_dict"
    ]
)

hybrid_model.eval()

print("Hybrid Loaded")

Hybrid Loaded


In [34]:
def calculate_metrics(
    targets,
    predictions
):

    acc = accuracy_score(
        targets,
        predictions
    )

    precision = precision_score(
        targets,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        targets,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        targets,
        predictions,
        zero_division=0
    )

    iou = jaccard_score(
        targets,
        predictions,
        zero_division=0
    )

    dice = (
        2 * precision * recall
    ) / (
        precision + recall + 1e-8
    )

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Dice": dice,
        "IoU": iou
    }

In [35]:
baseline_preds = []
baseline_targets = []

In [41]:
baseline_model.eval()

with torch.no_grad():

    for images, masks, sam_features in tqdm(test_loader):

        images = images.to(DEVICE)

        outputs = baseline_model(images)

        preds = (
            torch.sigmoid(outputs) > 0.5
        ).float()

        baseline_preds.extend(
            preds.cpu().numpy().flatten()
        )

        baseline_targets.extend(
            masks.cpu().numpy().flatten()
        )

  0%|          | 0/34 [00:00<?, ?it/s]

 91%|█████████ | 31/34 [00:15<00:01,  1.94it/s]


IndexError: list index out of range

In [42]:
print("Images:", len(test_dataset.images))
print("Masks:", len(test_dataset.masks))
print("SAM:", len(test_dataset.sam_features))

print("Dataset Length:", len(test_dataset))

Images: 268
Masks: 268
SAM: 254
Dataset Length: 268


In [43]:
image_names = {
    x.stem
    for x in test_dataset.images
}

sam_names = {
    x.stem
    for x in test_dataset.sam_features
}

missing = sorted(
    image_names - sam_names
)

print(
    "Missing SAM Files:",
    len(missing)
)

for m in missing:
    print(m)

Missing SAM Files: 14
EMSR292_01CHRISOUPOLI_10_07_2_1
EMSR326_01MORONDELAFRONTERA_02_19_2_1
EMSR326_02ANTEQUERA_02_11_2_2
EMSR326_02ANTEQUERA_09_01_2_2
EMSR326_02ANTEQUERA_09_09_1_2
EMSR326_02ANTEQUERA_13_11_2_2
EMSR326_02ANTEQUERA_14_05_2_2
EMSR326_02ANTEQUERA_15_07_1_2
EMSR326_02ANTEQUERA_16_08_1_1
EMSR326_02ANTEQUERA_17_04_1_1
EMSR326_02ANTEQUERA_17_04_2_2
EMSR326_02ANTEQUERA_17_05_2_1
EMSR326_04MALAGA_02_02_1_1
EMSR326_04MALAGA_03_01_1_2
